### Get market indices from 2008-2025

In [8]:
# necessary libraries
import yfinance as yf
import pandas as pd
import numpy as np
from datetime import datetime

In [ ]:
# date for which data needs to be downloaded
START_DATE = "2008-01-01"
END_DATE = "2025-01-01"

# Yahoo Finance tickers
TICKERS = {
    "sp500": "^GSPC",
    "vix": "^VIX",
    "10y_treasury": "^TNX",
    "2y_treasury": "^IRX"
}

# path to keep the data
OUTPUT_PATH = "../src/data/raw/market_data.csv"

In [10]:
# download the data of all the tickers
def download_market_data():

    print("Downloading market data...")

    data = yf.download(
        list(TICKERS.values()),
        start=START_DATE,
        end=END_DATE,
        auto_adjust=False,
        progress=True
    )

    return data

In [11]:
# convert to a dataframe with required columns
def process_market_data(data):

    df = pd.DataFrame()

    # S&P500
    df["sp500_open"] = data["Open"]["^GSPC"]
    df["sp500_high"] = data["High"]["^GSPC"]
    df["sp500_low"] = data["Low"]["^GSPC"]
    df["sp500_close"] = data["Close"]["^GSPC"]
    df["sp500_volume"] = data["Volume"]["^GSPC"]

    # VIX
    df["vix"] = data["Close"]["^VIX"]

    # Treasury Yields
    df["treasury_10y"] = data["Close"]["^TNX"]
    df["treasury_2y"] = data["Close"]["^IRX"]

    # calculating log returns
    df["sp500_log_return"] = np.log(
        df["sp500_close"] / df["sp500_close"].shift(1)
    )

    # calculate realized volatility for 20 day period
    df["realized_volatility"] = (
        df["sp500_log_return"]
        .rolling(window=20)
        .std()
    )

    df.reset_index(inplace=True)
    return df


In [12]:
# get the data and save it
raw_data = download_market_data()
market_df = process_market_data(raw_data)

print("Saving dataset...")

market_df.to_csv(OUTPUT_PATH, index=False)

print(f"Dataset saved at: {OUTPUT_PATH}")
print("Total rows:", len(market_df))

[*********************100%***********************]  4 of 4 completed

Saving dataset...
Dataset saved at: data/raw/market_data.csv
Total rows: 4279
